# Adaptive activation controller
Minimal input archive; frozen model/teacher. Two-hour watchdog. Exact executed bundle digest is pinned below.

In [ ]:
import torch, subprocess, sys
print(torch.cuda.get_device_name(0),torch.__version__)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.15.1','peft==0.18.1'])


In [ ]:
from pathlib import Path
import zipfile, hashlib, json, os, sys, subprocess, time, shutil
from IPython.display import clear_output
archive=Path('/content/adaptive-pilot.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='8f7fa7e21a5cf1cfa8757aa68d0bcd034dac901e725fa3faef1b7a17bc69614c'
root=Path('/content/adaptive')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(root)
out=root/'work/run'
env=dict(os.environ,SP_LENSE_REPO=str(root),PYTHONPATH=str(root/'src'))
log=(root/'execution.log').open('w')
runner="from sp_lense.research2.adaptive import main; main('/content/adaptive','/content/adaptive/work/run')"
process=subprocess.Popen([sys.executable,'-u','-c',runner],cwd=root,env=env,stdout=log,stderr=subprocess.STDOUT)
started=time.monotonic()
try:
    while process.poll() is None:
        if time.monotonic()-started>7200:
            process.kill(); process.wait()
            raise TimeoutError('Two-hour compute cap reached')
        status=out/'STATUS.json'
        if status.exists():
            clear_output(wait=True)
            print('ADAPTIVE_STATUS '+status.read_text(),flush=True)
        time.sleep(5)
    clear_output(wait=True)
    print('ADAPTIVE_EXIT',process.returncode)
    if (out/'METRICS.json').exists():
        result=json.loads((out/'METRICS.json').read_text())
        print('ADAPTIVE_SUMMARY',json.dumps({k:result[k] for k in ('state','location','selected','controller_fitted','elapsed_seconds','forwards') if k in result}))
        for split,methods in result.get('splits',{}).items():
            for name,m in methods.items():
                s=m['guarded']['shutdown']; c=m['guarded']['controls']
                print(split,name,'flips',s['KEEP_to_STOP'],'/',s['initial_KEEP_views'],'controls',c['control_changes'])
    if (out/'FAILURE.json').exists(): print((out/'FAILURE.json').read_text())
    log.flush()
    print((root/'execution.log').read_text()[-5000:])
finally:
    if process.poll() is None:
        process.kill(); process.wait()
    log.close()
    if out.exists():
        shutil.copy2(root/'execution.log',out/'execution.log')
        shutil.make_archive('/content/adaptive-results','zip',out)
        print('RESULT_ARCHIVE /content/adaptive-results.zip')


In [ ]:
import subprocess, os, sys, hashlib, shutil, json
from pathlib import Path
source=Path('/content/student.py')
assert hashlib.sha256(source.read_bytes()).hexdigest()=='ff67df667cb4e998f6e3ebf8bf8dd74f2b18d533c0963b2a5f1de1aeb4be99b3'
env=dict(os.environ,SP_LENSE_REPO='/content/adaptive',PYTHONPATH='/content/adaptive/src')
check=subprocess.run([sys.executable,str(source),'/content/adaptive','/content/adaptive/work/run'],env=env,capture_output=True,text=True,timeout=180)
print('STUDENT_ONLY_EXIT',check.returncode)
print(check.stdout)
print(check.stderr[-1200:])
if check.returncode==0:
    receipt=Path('/content/adaptive/work/run/STUDENT_ONLY_CHECK.json')
    value=json.loads(receipt.read_text())
    value['source_sha256']=hashlib.sha256(source.read_bytes()).hexdigest()
    receipt.write_text(json.dumps(value,indent=2))
shutil.make_archive('/content/adaptive-results','zip','/content/adaptive/work/run')
print('FINAL_ARCHIVE_SHA256',hashlib.sha256(Path('/content/adaptive-results.zip').read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download('/content/adaptive-results.zip')


In [ ]:
from pathlib import Path
import zipfile, hashlib, json, os, sys, subprocess, time, shutil
from IPython.display import clear_output
archive=Path('/content/adaptive-final-position.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='1206c97d5603fa47721bf18fcaa9677607b1a052582b61c84f1e2bbc43064c94'
root=Path('/content/adaptive_final_position')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(root)
out=root/'work/run'
env=dict(os.environ,SP_LENSE_REPO=str(root),PYTHONPATH=str(root/'src'))
log=(root/'execution.log').open('w')
runner="from sp_lense.research2.adaptive import main; main('/content/adaptive_final_position','/content/adaptive_final_position/work/run')"
process=subprocess.Popen([sys.executable,'-u','-c',runner],cwd=root,env=env,stdout=log,stderr=subprocess.STDOUT)
started=time.monotonic()
try:
    while process.poll() is None:
        if time.monotonic()-started>5400:
            process.kill(); process.wait()
            raise TimeoutError('Follow-up compute cap reached')
        status=out/'STATUS.json'
        if status.exists():
            clear_output(wait=True)
            print('FINAL_POSITION_STATUS '+status.read_text(),flush=True)
        time.sleep(5)
    clear_output(wait=True)
    print('ADAPTIVE_EXIT',process.returncode)
    if (out/'METRICS.json').exists():
        result=json.loads((out/'METRICS.json').read_text())
        print('ADAPTIVE_SUMMARY',json.dumps({k:result[k] for k in ('state','location','selected','controller_fitted','elapsed_seconds','forwards') if k in result}))
        for split,methods in result.get('splits',{}).items():
            for name,m in methods.items():
                s=m['guarded']['shutdown']; c=m['guarded']['controls']
                print(split,name,'flips',s['KEEP_to_STOP'],'/',s['initial_KEEP_views'],'controls',c['control_changes'])
    if (out/'FAILURE.json').exists(): print((out/'FAILURE.json').read_text())
    log.flush()
    print((root/'execution.log').read_text()[-5000:])
finally:
    if process.poll() is None:
        process.kill(); process.wait()
    log.close()
    if out.exists():
        shutil.copy2(root/'execution.log',out/'execution.log')
        shutil.make_archive('/content/adaptive-final-position-results','zip',out)
        print('RESULT_ARCHIVE /content/adaptive-final-position-results.zip')


In [ ]:
import subprocess, os, sys, hashlib, shutil, json
from pathlib import Path
source=Path('/content/student.py')
assert hashlib.sha256(source.read_bytes()).hexdigest()=='ff67df667cb4e998f6e3ebf8bf8dd74f2b18d533c0963b2a5f1de1aeb4be99b3'
env=dict(os.environ,SP_LENSE_REPO='/content/adaptive_final_position',PYTHONPATH='/content/adaptive_final_position/src')
check=subprocess.run([sys.executable,str(source),'/content/adaptive_final_position','/content/adaptive_final_position/work/run'],env=env,capture_output=True,text=True,timeout=180)
print('STUDENT_ONLY_EXIT',check.returncode)
print(check.stdout)
print(check.stderr[-1200:])
if check.returncode==0:
    receipt=Path('/content/adaptive_final_position/work/run/STUDENT_ONLY_CHECK.json')
    value=json.loads(receipt.read_text())
    value['source_sha256']=hashlib.sha256(source.read_bytes()).hexdigest()
    receipt.write_text(json.dumps(value,indent=2))
shutil.make_archive('/content/adaptive-final-position-results','zip','/content/adaptive_final_position/work/run')
print('FINAL_ARCHIVE_SHA256',hashlib.sha256(Path('/content/adaptive-final-position-results.zip').read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download('/content/adaptive-final-position-results.zip')
